# TP 5 — Graphes : représentation, parcours et plus courts chemins
### Algorithmique avancée et complexité — Master IA et Master RSD · Structures de données et graphes

| | |
|---|---|
| **Cours & TD** | Dr. Benyamina Ahmed |
| **TP** | Mahkouka Mohamed Fouad |
| **Durée** | 1 séance |

---
## 🎯 Objectif

Implémenter en Python un **graphe non orienté pondéré**, le manipuler, réaliser les parcours **BFS** et **DFS**, et calculer les **plus courts chemins** avec l'algorithme de **Dijkstra**.

| Partie | Contenu |
|---|---|
| 1 | Le graphe et ses données |
| 2 | Représentation (liste d'adjacence pondérée) et manipulation |
| 3 | Parcours en largeur (BFS) et en profondeur (DFS) |
| 4 | Plus courts chemins : Dijkstra et reconstruction du chemin |
| 5 | Programme principal, tests et performances |

Dans chaque partie : ✍️ vous **programmez** les fonctions, ✅ une cellule **vérifie** qu'elles sont justes.

**Contraintes :** Python 3 uniquement, **pas de bibliothèque externe** pour les algorithmes (`collections`, `heapq` et `time` font partie de Python), fonctions modulaires et commentées, résultats vérifiés par des tests.

---
# Outils

**Exécutez cette cellule en premier.** `verifier` affiche ✅ ou ❌ ; `chrono_ms(f, *args)` donne le temps **moyen** en millisecondes sur 100 exécutions.

In [ ]:
import heapq
import time
from collections import deque


def verifier(nom, obtenu, attendu):
    print(("✅" if obtenu == attendu else "❌"), nom, "| obtenu :", obtenu, "| attendu :", attendu)


def chrono_ms(f, *args, repetitions=100):
    # Temps moyen de f(*args) en millisecondes, sur 100 exécutions.
    debut = time.perf_counter()
    for _ in range(repetitions):
        f(*args)
    return (time.perf_counter() - debut) / repetitions * 1000


print("Outils prêts ✅")

---
# Partie 1 — Le graphe et ses données *(fourni)*

Graphe **non orienté pondéré** à 7 sommets et 11 arêtes (voir la figure de la fiche) :


| # | Arête (u, v) | Poids w | | # | Arête (u, v) | Poids w |
|---|---|---|---|---|---|---|
| 1 | (A, B) | 4 | | 7 | (C, F) | 2 |
| 2 | (A, D) | 8 | | 8 | (D, E) | 10 |
| 3 | (A, E) | 1 | | 9 | (E, F) | 7 |
| 4 | (B, E) | 3 | | 10 | (E, G) | 6 |
| 5 | (B, C) | 2 | | 11 | (F, G) | 4 |
| 6 | (C, E) | 8 | | | | |

> **Rappel :** un graphe $G = (V, E)$. Dans une **liste d'adjacence**, on associe à chaque sommet $u$ la liste de ses voisins avec le poids de l'arête : `G = {"A": [("B", 4), ("D", 8), ("E", 1)], ...}`.

In [ ]:
ARETES = [("A", "B", 4), ("A", "D", 8), ("A", "E", 1), ("B", "E", 3), ("B", "C", 2), ("C", "E", 8),
          ("C", "F", 2), ("D", "E", 10), ("E", "F", 7), ("E", "G", 6), ("F", "G", 4)]

---
# Partie 2 — Représentation et manipulation

Le graphe est un **dictionnaire** : sommet → liste de `(voisin, poids)`.

✍️ Implémentez :
- `ajouter_sommet(G, s)` : ajoute `s` avec une liste vide **s'il n'existe pas déjà** ;
- `ajouter_arete(G, u, v, w)` : ajoute les deux sommets si besoin, puis l'arête **dans les deux sens** (graphe non orienté) ;
- `afficher_graphe(G)` : affiche chaque arête **une seule fois**, sous la forme `A — B (4)` ;
- `afficher_liste_adjacence(G)` : affiche une ligne par sommet, `A : [('B', 4), ...]` ;
- `construire_graphe(aretes)` : construit le graphe à partir de la liste `ARETES`.

In [ ]:
def ajouter_sommet(G, s):
    # Ajoute le sommet s (sans voisin) s'il n'existe pas encore.
    pass  # À COMPLÉTER


def ajouter_arete(G, u, v, w):
    # Graphe non orienté : l'arête u — v de poids w est stockée dans les deux sens.
    pass  # À COMPLÉTER (utilisez ajouter_sommet)


def afficher_graphe(G):
    # Affiche chaque arête une seule fois, ex. « A — B (4) ».
    # Astuce : mémorisez dans un set les arêtes (u, v) déjà affichées, et sautez (v, u).
    pass  # À COMPLÉTER


def afficher_liste_adjacence(G):
    # Une ligne par sommet : « A : [('B', 4), ('D', 8), ('E', 1)] ».
    pass  # À COMPLÉTER


def construire_graphe(aretes):
    # Construit le graphe (dictionnaire) à partir d'une liste de (u, v, w).
    G = {}
    # À COMPLÉTER
    return G

In [ ]:
G = construire_graphe(ARETES)
afficher_liste_adjacence(G)
print()
afficher_graphe(G)
print()
verifier("nombre de sommets", len(G), 7)
verifier("nombre d'arêtes", sum(len(v) for v in G.values()) // 2, 11)
verifier("voisins de A", G.get("A"), [("B", 4), ("D", 8), ("E", 1)])
verifier("voisins de E", G.get("E"), [("A", 1), ("B", 3), ("C", 8), ("D", 10), ("F", 7), ("G", 6)])
ajouter_sommet(G, "A")
verifier("ajouter_sommet ne vide pas un sommet existant", G.get("A"), [("B", 4), ("D", 8), ("E", 1)])

> ✍️ **Pourquoi le nombre d'arêtes est-il la somme des longueurs des listes divisée par 2 ?** …

---
# Partie 3 — Parcours du graphe

**BFS (en largeur)** : on visite la source, puis tous ses voisins, puis les voisins des voisins… avec une **file FIFO** (`deque`) : `popleft()` pour retirer en tête, `append()` pour ajouter en queue. Un sommet est marqué **visité au moment où on l'ajoute** dans la file (il n'y entre qu'une fois).

**DFS (en profondeur)** : on part d'un sommet et on va le plus loin possible avant de revenir en arrière. Version **récursive** : marquer `s` comme visité, l'ajouter à l'ordre, puis appeler `dfs` sur chaque voisin pas encore visité.

⚠️ `G[u]` contient des **couples** `(voisin, poids)` : écrivez `for v, w in G[u]` (et non `for v in G[u]`).

Les voisins sont examinés **dans l'ordre de la liste d'adjacence**.

✍️ Implémentez `bfs(G, depart)` et `dfs(G, s, visites=None, ordre=None)` ; les deux renvoient la liste des sommets dans l'**ordre de visite**.

In [ ]:
def bfs(G, depart):
    # Parcours en largeur ; renvoie l'ordre de visite.
    visites = {depart}
    ordre = []
    file = deque([depart])          # file FIFO
    # À COMPLÉTER : tant que la file n'est pas vide,
    #   retirer u en tête, l'ajouter à ordre,
    #   puis ajouter en queue ses voisins pas encore visités (et les marquer visités)
    return ordre


def dfs(G, s, visites=None, ordre=None):
    # Parcours en profondeur récursif ; renvoie l'ordre de visite (préordre).
    if visites is None:
        visites = set()
    if ordre is None:
        ordre = []
    # À COMPLÉTER : marquer s, l'ajouter à ordre, appeler dfs sur chaque voisin non visité
    return ordre

In [ ]:
print("BFS depuis A :", " → ".join(bfs(G, "A")))
print("DFS depuis A :", " → ".join(dfs(G, "A")))
verifier("BFS depuis A", bfs(G, "A"), ["A", "B", "D", "E", "C", "F", "G"])
verifier("DFS depuis A", dfs(G, "A"), ["A", "B", "E", "C", "F", "G", "D"])
verifier("tous les sommets visités une seule fois", sorted(dfs(G, "A")), ["A", "B", "C", "D", "E", "F", "G"])

> ✍️ **En DFS, pourquoi visite-t-on E juste après B, et pourquoi D en dernier ?** …

---
# Partie 4 — Plus courts chemins : Dijkstra

**Principe :**
- initialiser $d(A) = 0$ et $d(v) = \infty$ pour $v \neq A$ ;
- à chaque itération, choisir le sommet **non traité** de plus petite distance (file de priorité `heapq`) ;
- mettre à jour les distances de ses voisins :

```
si d + w < dist[v] :
    dist[v] ← d + w ; pred[v] ← u ; ajouter (dist[v], v) dans la file
```

`pred[v]` est le **prédécesseur** de `v` sur le plus court chemin.

**Reconstruction du chemin :** partir de `destination`, remonter avec `pred[courant]` jusqu'à `None`, puis **inverser** la liste. Si le chemin ne commence pas par `source` (sommet inaccessible), renvoyer `None`.

✍️ Implémentez `dijkstra(G, source)` → `(dist, pred)` et `reconstruire_chemin(pred, source, destination)`.

In [ ]:
def dijkstra(G, source):
    # Renvoie (dist, pred) : distance minimale depuis source et prédécesseur de chaque sommet.
    dist = {v: float("inf") for v in G}
    pred = {v: None for v in G}
    dist[source] = 0
    pq = [(0, source)]              # file de priorité (distance, sommet)
    traites = set()
    while pq:
        d, u = heapq.heappop(pq)
        if u in traites:
            continue                # entrée périmée : u déjà traité
        traites.add(u)
        # À COMPLÉTER : pour chaque voisin (v, w) de u,
        #   si d + w < dist[v] : mettre à jour dist[v] et pred[v], puis pousser (dist[v], v) dans pq
    return dist, pred


def reconstruire_chemin(pred, source, destination):
    # Remonte les prédécesseurs depuis destination, puis inverse la liste.
    chemin = []
    # À COMPLÉTER (renvoyer None si le chemin ne commence pas par source)
    return chemin

In [ ]:
dist, pred = dijkstra(G, "A")
verifier("distances depuis A", dist, {"A": 0, "B": 4, "C": 6, "D": 8, "E": 1, "F": 8, "G": 7})
verifier("prédécesseurs", pred, {"A": None, "B": "A", "C": "B", "D": "A", "E": "A", "F": "E", "G": "E"})
verifier("chemin A → G", reconstruire_chemin(pred, "A", "G"), ["A", "E", "G"])
verifier("chemin A → C", reconstruire_chemin(pred, "A", "C"), ["A", "B", "C"])
verifier("chemin A → A", reconstruire_chemin(pred, "A", "A"), ["A"])

# Sommet isolé H : aucun chemin depuis A
G2 = construire_graphe(ARETES)
ajouter_sommet(G2, "H")
verifier("chemin A → H (inaccessible)", reconstruire_chemin(dijkstra(G2, "A")[1], "A", "H"), None)

print(f"\n{'Sommet':<7} {'Plus court chemin':<20} {'d(v)':>5}")
for v in G:
    chemin = reconstruire_chemin(pred, "A", v) or []
    print(f"{v:<7} {' → '.join(chemin):<20} {dist[v]:>5}")

> ✍️ **Vers F, deux chemins coûtent 8 (A → E → F et A → B → C → F). Pourquoi Dijkstra garde-t-il A → E → F ?** …

> ✍️ **Les arêtes `(pred[v], v)` forment l'arbre des plus courts chemins. Lesquelles ? Combien y en a-t-il ?** …

---
# Partie 5 — Programme principal, tests et performances

Le programme principal regroupe toutes les fonctions sur le graphe du TP, puis mesure le **temps moyen** de BFS, DFS et Dijkstra sur **100 exécutions**. *(Exécutez seulement.)*

In [ ]:
G = construire_graphe(ARETES)
print("Graphe (liste d'adjacence) :")
afficher_liste_adjacence(G)

ordre_bfs = bfs(G, "A")
ordre_dfs = dfs(G, "A")
dist, pred = dijkstra(G, "A")
chemin = reconstruire_chemin(pred, "A", "G")
print("\nBFS (depuis A) :", ordre_bfs)
print("DFS (depuis A) :", ordre_dfs)
print("Distances minimales :", dist)
print("Prédécesseurs :", pred)
print("Chemin A → G :", chemin, "| coût :", dist.get("G"))

# Tableau des résultats
tests = [
    ("Ordre BFS (depuis A)", ordre_bfs, ["A", "B", "D", "E", "C", "F", "G"]),
    ("Ordre DFS (depuis A)", ordre_dfs, ["A", "B", "E", "C", "F", "G", "D"]),
    ("Distance minimale d(A, G)", dist.get("G"), 7),
    ("Chemin minimal A → G", chemin, ["A", "E", "G"]),
]
print(f"\n{'Élément testé':<28} {'Résultat obtenu':<38} Statut")
for nom, obtenu, attendu in tests:
    resultat = " → ".join(obtenu) if isinstance(obtenu, list) else str(obtenu)
    print(f"{nom:<28} {resultat:<38} {'✅ Validé' if obtenu == attendu else '❌'}")

# Performances (moyenne de 100 exécutions)
print(f"\n{'Algorithme':<10} {'Complexité':<20} {'Temps moyen (ms)':>17}")
for nom, cplx, f in [("BFS", "O(V + E)", bfs), ("DFS", "O(V + E)", dfs), ("Dijkstra", "O((V + E) log V)", dijkstra)]:
    print(f"{nom:<10} {cplx:<20} {chrono_ms(f, G, 'A'):>17.4f}")

> ✍️ **Pourquoi Dijkstra est-il le plus lent des trois ?** …

---
# 📤 Travail à rendre

1. Ce notebook complété : **Restart & Run All** sans erreur, toutes les vérifications ✅.
2. Les lignes ✍️ complétées (elles constituent l'analyse des résultats).
3. Envoyez sur GitHub :

```bash
git add tp5_graphes/
git commit -m "TP5 : graphes, parcours et Dijkstra"
git push
```

## 🎯 À retenir

| Algorithme | Rôle | Structure clé | Complexité |
|---|---|---|---|
| BFS | parcours par niveaux | file FIFO (`deque`) | $O(V + E)$ |
| DFS | parcours en profondeur | récursion (pile LIFO) | $O(V + E)$ |
| Dijkstra | plus courts chemins (poids ≥ 0) | file de priorité (`heapq`) | $O((V + E) \log V)$ |

- Liste d'adjacence d'un graphe non orienté : chaque arête est stockée **deux fois**.
- L'ordre de visite dépend de l'**ordre des voisins** dans la liste d'adjacence.
- Dijkstra ne fonctionne qu'avec des **poids positifs** ; `pred` permet de **reconstruire** le chemin.